# ĐỒ ÁN XỬ LÝ NGÔN NGỮ TỰ NHIÊN
## Tóm tắt văn bản bằng phương pháp phân lớp (7 đặc trưng + Logistic Regression)

7 đặc trưng = 5 đặc trưng của Kupiec (slide 13) + **Quote** (câu trích dẫn) + **Số lượng** (câu có con số chỉ số lượng).

In [118]:
# ===== KHAI BÁO THƯ VIỆN =====
import os                          # làm việc với đường dẫn, thư mục
import re                          # biểu thức chính quy (regex) để tìm chuỗi
import html                        # đổi ký tự HTML như &amp; thành &
import csv                         # ghi file CSV
from nltk.stem import PorterStemmer                              # đưa từ tiếng Anh về từ gốc
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # danh sách stopword tiếng Anh
from sklearn.linear_model import LogisticRegression

print("Đã nạp đủ thư viện.")

Đã nạp đủ thư viện.


In [119]:
class Cau:
    def __init__(self, tap, cum, ma_bai, stt, so_tu, cau_goc):
        self.tap = tap            # train hay test
        self.cum = cum            # tên file = mã cụm, ví dụ d061j
        self.ma_bai = ma_bai      # mã bài báo
        self.stt = stt            # số thứ tự câu trong bài
        self.so_tu = so_tu        # số từ của câu (wdcount)
        self.cau_goc = cau_goc    # nội dung câu gốc

class Cum:
    def __init__(self, ma, tap, ds_cau):
        self.ma = ma              # mã cụm = tên file, ví dụ d061j
        self.tap = tap            # train hay test
        self.ds_cau = ds_cau      # list các câu (object Cau) của cụm

# ===== HÀM IN =====
def in_bang(danh_sach_cau, cac_thuoc_tinh):
    """In bảng: mỗi dòng là 1 câu, mỗi cột là 1 thuộc tính của câu."""
    tieu_de = ""
    for thuoc_tinh in cac_thuoc_tinh:
        tieu_de = tieu_de + thuoc_tinh + " | "
    print(tieu_de)

    for i in range(5):
        cau = danh_sach_cau[i]
        dong = ""
        for thuoc_tinh in cac_thuoc_tinh:
            gia_tri = str(getattr(cau, thuoc_tinh))
            if len(gia_tri) > 60:            # tối đa 60 kí tự
                gia_tri = gia_tri[0:60] + "..."
            dong = dong + gia_tri + " | "
        print(dong)


## 3.1. Đọc dữ liệu

In [120]:
# ===== ĐỌC DỮ LIỆU =====
THU_MUC_VAN_BAN = os.path.join('.', 'DUC_TEXT')      # văn bản gốc

# <s docid="AP880911-0016" num="9" wdcount="28"> Hurricane Gilbert swept toward ... </s>
#           mã bài báo         stt         so_tu      nội dung câu
MAU_CAU = re.compile(r'<s docid="(.*?)" num="(\d+)" wdcount="(\d+)">(.*?)</s>', re.S)

def doc_file_duc(duong_dan, tap, cum):
    """Đọc 1 file DUC, trả về danh sách câu (mỗi câu là một object Cau)."""
    f = open(duong_dan, encoding='utf-8', errors='ignore')
    toan_bo = f.read()
    f.close()
    ds = []
    for ma_bai, stt, so_tu, noi_dung in MAU_CAU.findall(toan_bo):
        ds.append(Cau(tap, cum, ma_bai, int(stt), int(so_tu), noi_dung.strip()))
    return ds # list câu của 1 file

# Đọc tất cả file trong DUC_TEXT/train và DUC_TEXT/test
ds_cau = []        
ds_cum = []       
for tap in ['train', 'test']:
    thu_muc = os.path.join(THU_MUC_VAN_BAN, tap)
    for ten_file in sorted(os.listdir(thu_muc)):
        if ten_file.startswith('.'):           # bỏ file ẩn của hệ điều hành
            continue
        duong_dan = os.path.join(thu_muc, ten_file)
        cum = Cum(ten_file, tap, doc_file_duc(duong_dan, tap, ten_file))   # tên file = mã cụm
        ds_cum.append(cum)
        ds_cau += cum.ds_cau                

# # # Đếm số cụm và số bài báo
# # cum_train, cum_test, cac_bai = set(), set(), set()
# # for cau in ds_cau:
# #     if cau.tap == 'train':
# #         cum_train.add(cau.cum)
# #     else:
# #         cum_test.add(cau.cum)
# #     cac_bai.add((cau.cum, cau.ma_bai))

# # print(f"{len(ds_cum)} cụm (train {len(cum_train)}, test {len(cum_test)}), "
# #       f"{len(cac_bai)} bài báo, {len(ds_cau)} câu")
# # print("Câu đầu tiên:", vars(ds_cau[0]))

## 3.2. Gán nhãn từ DUC_SUM

In [121]:
# ===== GÁN NHÃN TỪ DUC_SUM =====
THU_MUC_DAP_AN  = os.path.join('.', 'DUC_SUM')

# Bước 1: với mỗi cụm, đọc đáp án và gom các cặp (ma_bai, stt)
cum_rong = []                      
for cum in ds_cum:
    cum.dap_an = doc_file_duc(os.path.join(THU_MUC_DAP_AN, cum.ma), None, cum.ma)
    cum.cau_chuyen_gia = set()       # {('AP880911-0016', 9), ...}
    for cau_cg in cum.dap_an:
        cum.cau_chuyen_gia.add((cau_cg.ma_bai, cau_cg.stt))
    if len(cum.dap_an) == 0:
        cum_rong.append(cum.ma)

# Bước 2: gán nhãn cho từng câu
dem_nhan = {1: 0, 0: 0, None: 0}
for cum in ds_cum:
    for cau in cum.ds_cau:
        if len(cum.dap_an) == 0:
            cau.nhan = None
        elif (cau.ma_bai, cau.stt) in cum.cau_chuyen_gia:
            cau.nhan = 1
        else:
            cau.nhan = 0
        dem_nhan[cau.nhan] += 1

tong_cau_chuyen_gia = 0
for cum in ds_cum:
    tong_cau_chuyen_gia += len(cum.dap_an)
    
print(f"{len(ds_cum) - len(cum_rong)} cụm có DUC_SUM, {tong_cau_chuyen_gia} câu tóm tắt của chuyên gia")
print("Cụm có DUC_SUM rỗng:", cum_rong)
print("Số câu theo nhãn (1 = tóm tắt, 0 = thường, None = không có nhãn):", dem_nhan)

43 cụm có DUC_SUM, 638 câu tóm tắt của chuyên gia
Cụm có DUC_SUM rỗng: ['d062j', 'd067f', 'd068f', 'd071f', 'd072f', 'd074b', 'd078b', 'd084a', 'd085d', 'd086d', 'd090d', 'd097e', 'd098e', 'd104g', 'd106g', 'd108g']
Số câu theo nhãn (1 = tóm tắt, 0 = thường, None = không có nhãn): {1: 588, 0: 12542, None: 4210}


## 3.3. Tiền xử lý văn bản

In [122]:
NEWS_STOP_WORDS = {'said', 'say', 'says', 'told', 'also', 'would', 'could', 'year', 'years'}
PATTERN_TACH_TU = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|[A-Za-z0-9]+(?:[.'-][A-Za-z0-9]+)*")
bo_dua_ve_goc = PorterStemmer()

def lam_sach(cau_van):
    cau_van = html.unescape(cau_van)                         # &amp; thành &
    cau_van = cau_van.replace('``', '"').replace("''", '"')  # ``abc'' thành "abc"
    cau_van = re.sub(r'\s+', ' ', cau_van)                   # nhiều khoảng trắng thành 1
    return cau_van.strip()

# loại bỏ số, bỏ stopword, rồi đưa về từ gốc
def lay_tu_noi_dung(cac_tu):
    ket_qua = []
    for tu in cac_tu:
        if not tu.isalpha() or len(tu) <= 1:
            continue
        tu_thuong = tu.lower()
        if tu_thuong in ENGLISH_STOP_WORDS:
            continue
        if tu_thuong in NEWS_STOP_WORDS:
            continue
        ket_qua.append(bo_dua_ve_goc.stem(tu_thuong))
    return ket_qua

for cau in ds_cau:
    cau.cau_sach = lam_sach(cau.cau_goc)
    cau.cac_tu = PATTERN_TACH_TU.findall(cau.cau_sach)
    cau.tu_noi_dung = lay_tu_noi_dung(cau.cac_tu)

vi_du = ds_cau[2]
print("Câu gốc      :", vi_du.cau_goc)
print("Sau làm sạch :", vi_du.cau_sach)
print("Tách từ      :", vi_du.cac_tu)
print("Từ nội dung  :", vi_du.tu_noi_dung)

Câu gốc      : ``There is no need for alarm,'' Civil Defense Director Eugenio Cabral said in a television alert shortly before midnight Saturday.
Sau làm sạch : "There is no need for alarm," Civil Defense Director Eugenio Cabral said in a television alert shortly before midnight Saturday.
Tách từ      : ['There', 'is', 'no', 'need', 'for', 'alarm', 'Civil', 'Defense', 'Director', 'Eugenio', 'Cabral', 'said', 'in', 'a', 'television', 'alert', 'shortly', 'before', 'midnight', 'Saturday']
Từ nội dung  : ['need', 'alarm', 'civil', 'defens', 'director', 'eugenio', 'cabral', 'televis', 'alert', 'shortli', 'midnight', 'saturday']


## 4.1. Đặc trưng 1: Sentence Length Cut-off

In [123]:
# ===== ĐẶC TRƯNG 1: SENTENCE LENGTH =====
LENGTH_CUTOFF = 5   

dem_cau_ngan = 0
for cau in ds_cau:
    if cau.so_tu > LENGTH_CUTOFF:         
        cau.dt1_do_dai = 1       
    else:
        cau.dt1_do_dai = 0 
        dem_cau_ngan += 1

print(f"Câu có dưới {LENGTH_CUTOFF} từ: {dem_cau_ngan} câu ({dem_cau_ngan / len(ds_cau):.1%})")
# # cau_ngan = []
# # for cau in ds_cau:
# #     if cau.dt1_do_dai == 0:
# #         cau_ngan.append(cau)
# # in_bang(cau_ngan, ['cum', 'cau_sach', 'so_tu'])

Câu có dưới 5 từ: 1950 câu (11.2%)


## 4.2. Đặc trưng 2: Fixed-Phrase

In [124]:
# ===== ĐẶC TRƯNG 2: FIXED-PHRASE =====
FIXED_PHRASE = [
    'this letter', 'in conclusion', 'in summary', 'to sum up', 'to summarize', 'in short', 'in brief', 'we conclude',
    'according to', 'officials said', 'the report', 'announced', 'for the first time',
    'the worst', 'the largest', 'the biggest', 'the most', 'is expected to', 'overall', 'finally']

def tim_cum_tu(cau_van):
    cau_thuong = cau_van.lower()
    tim_thay = []
    for cum_tu in FIXED_PHRASE:
        if re.search(r'\b' + cum_tu + r'\b', cau_thuong):
            tim_thay.append(cum_tu)
    return tim_thay

dem_moi_cum_tu = {}                  # đếm mỗi cụm từ xuất hiện trong bao nhiêu câu
for cum_tu in FIXED_PHRASE:
    dem_moi_cum_tu[cum_tu] = 0

dem_cau_co_cum_tu = 0
for cau in ds_cau:
    tim_thay = tim_cum_tu(cau.cau_sach)
    cau.cum_tu_tim_thay = tim_thay
    if len(tim_thay) > 0:
        cau.dt2_cum_tu = 1
        dem_cau_co_cum_tu += 1
    else:
        cau.dt2_cum_tu = 0
    for cum_tu in tim_thay:
        dem_moi_cum_tu[cum_tu] += 1

print(f"Câu có cụm từ cố định: {dem_cau_co_cum_tu} ({dem_cau_co_cum_tu / len(ds_cau):.1%})")
for cum_tu in FIXED_PHRASE:
    print(f"  {cum_tu:<20}: {dem_moi_cum_tu[cum_tu]} câu")

Câu có cụm từ cố định: 686 (4.0%)
  this letter         : 0 câu
  in conclusion       : 0 câu
  in summary          : 0 câu
  to sum up           : 0 câu
  to summarize        : 0 câu
  in short            : 0 câu
  in brief            : 0 câu
  we conclude         : 0 câu
  according to        : 116 câu
  officials said      : 121 câu
  the report          : 17 câu
  announced           : 89 câu
  for the first time  : 24 câu
  the worst           : 45 câu
  the largest         : 39 câu
  the biggest         : 25 câu
  the most            : 146 câu
  is expected to      : 34 câu
  overall             : 23 câu
  finally             : 29 câu


## 4.3. Đặc trưng 3: Paragraph

In [125]:
# ===== ĐẶC TRƯNG 3: PARAGRAPH =====
# Bước 1: gom số thứ tự câu (stt) của từng bài báo
stt_cua_bai = {}                     # ('d061j', 'AP880911-0016'): [9, 10, 11, ..., 24]
for cau in ds_cau:
    khoa = (cau.cum, cau.ma_bai)
    if khoa not in stt_cua_bai:
        stt_cua_bai[khoa] = []
    stt_cua_bai[khoa].append(cau.stt)

for khoa in stt_cua_bai:             
    stt_cua_bai[khoa].sort()

# Bước 2: xác định vị trí của từng câu trong bài
dem_vi_tri = {'INITIAL': 0, 'MEDIAL': 0, 'FINAL': 0}   
for cau in ds_cau:
    cac_stt = stt_cua_bai[(cau.cum, cau.ma_bai)]
    cau.vi_tri = cac_stt.index(cau.stt) + 1 
    cau.so_cau_cua_bai = len(cac_stt)          

    cau.dt3_vi_tri = 'MEDIAL'
    if cau.vi_tri == 1:
        cau.dt3_vi_tri = 'INITIAL'                 
    elif cau.vi_tri == cau.so_cau_cua_bai:
        cau.dt3_vi_tri = 'FINAL' 

    dem_vi_tri[cau.dt3_vi_tri] += 1

    # Bước 3: tách Paragraph thành 2 cột 0/1 để đưa vào LR
    cau.la_cau_dau  = 1 if cau.dt3_vi_tri == 'INITIAL' else 0
    cau.la_cau_cuoi = 1 if cau.dt3_vi_tri == 'FINAL' else 0

print(dem_vi_tri)

{'INITIAL': 567, 'MEDIAL': 16206, 'FINAL': 567}


## 4.4. Đặc trưng 4: Thematic Word

In [126]:
# ===== ĐẶC TRƯNG 4: THEMATIC WORD =====
NUM_THEMATIC_WORD = 10   
MIN_THEMATIC_WORD = 2  

def lay_so_lan(cap): #('storm', 50)
    return cap[1]

# Bước 1: với mỗi cụm, lấy 10 từ xuất hiện nhiều nhất làm từ chủ đề
for cum in ds_cum:
    dem_tu = {} 
    for cau in cum.ds_cau:
        for tu in cau.tu_noi_dung:
            if tu not in dem_tu:
                dem_tu[tu] = 0
            dem_tu[tu] += 1 # {'storm': 50, 'hurrican': 80, 'wind': 30}

    cac_cap = sorted(dem_tu.items(), key=lay_so_lan, reverse=True) # ('storm', 50), ('hurrican', 80), ('wind', 30)
    cum.tu_chu_de = []
    for tu, so_lan in cac_cap[:NUM_THEMATIC_WORD]:
        cum.tu_chu_de.append(tu)

# Bước 2: câu chứa ít nhất 2 từ chủ đề khác nhau thì Thematic = 1
dem_cau_chu_de = 0
for cum in ds_cum:
    for cau in cum.ds_cau:
        tim_thay = set(cau.tu_noi_dung) & set(cum.tu_chu_de)
        cau.tu_chu_de_tim_thay = tim_thay
        if len(tim_thay) >= MIN_THEMATIC_WORD:
            cau.dt4_chu_de = 1
            dem_cau_chu_de += 1
        else:
            cau.dt4_chu_de = 0

print(f"Câu có Thematic = 1: {dem_cau_chu_de} ({dem_cau_chu_de / len(ds_cau):.1%})")

# # for cum in ds_cum:
# #     if cum.ma == 'd061j':
# #         cum_vi_du = cum
# # cau_d061j = cum_vi_du.ds_cau                   

# # print("Từ chủ đề của cụm d061j (bão Gilbert):", cum_vi_du.tu_chu_de)
# # in_bang(cau_d061j, ['cau_sach', 'tu_chu_de_tim_thay', 'dt4_chu_de'])

Câu có Thematic = 1: 6183 (35.7%)


## 4.5. Đặc trưng 5: Uppercase Word

In [127]:
# ===== ĐẶC TRƯNG 5: UPPERCASE WORD =====
NONE_UPPERCASE = {'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday',
                  'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August',
                  'September', 'October', 'November', 'December', 'Mr', 'Mrs', 'Ms', 'Dr'}
SO_TEN_RIENG = 10

def lay_tu_viet_hoa(cac_tu):
    ket_qua = []
    for tu in cac_tu[1:]:                    
        if not tu[0].isupper():               
            continue
        if not tu.isalpha() or len(tu) < 2: 
            continue
        if tu.lower() in ENGLISH_STOP_WORDS or tu in NONE_UPPERCASE:
            continue
        if tu.isupper():                      # MICHAEL thành Michael
            tu = tu.title()
        ket_qua.append(tu)
    return ket_qua

# Bước 1 và 2: với mỗi cụm, lấy từ viết hoa của từng câu, đếm, rồi lấy 10 tên xuất hiện nhiều nhất
for cum in ds_cum:
    dem_ten = {} 
    for cau in cum.ds_cau:
        cau.tu_viet_hoa = lay_tu_viet_hoa(cau.cac_tu)
        for tu in cau.tu_viet_hoa:
            if tu not in dem_ten:
                dem_ten[tu] = 0
            dem_ten[tu] += 1 
    cac_cap = sorted(dem_ten.items(), key=lay_so_lan, reverse=True)
    cum.ten_rieng_chinh = []
    for tu, so_lan in cac_cap[:SO_TEN_RIENG]:
        cum.ten_rieng_chinh.append(tu)

# Bước 3: câu chứa ít nhất 1 tên riêng chính thì Uppercase = 1
dem_cau_ten_rieng = 0
for cum in ds_cum:
    for cau in cum.ds_cau:
        tim_thay = set(cau.tu_viet_hoa) & set(cum.ten_rieng_chinh)
        if len(tim_thay) > 0:
            cau.dt5_ten_rieng = 1
            dem_cau_ten_rieng += 1
        else:
            cau.dt5_ten_rieng = 0

# # print("Tên riêng chính của cụm d061j:", cum_vi_du.ten_rieng_chinh)
print(f"Câu có Uppercase = 1: {dem_cau_ten_rieng} ({dem_cau_ten_rieng / len(ds_cau):.1%})")

Câu có Uppercase = 1: 7226 (41.7%)


## 4.6. Đặc trưng 6: Quote

In [128]:
# ===== ĐẶC TRƯNG 6: QUOTE =====
dem_cau_trich_dan = 0
for cau in ds_cau:
    if '"' in cau.cau_sach:
        cau.dt6_trich_dan = 1
        dem_cau_trich_dan +=1
    else:
        cau.dt6_trich_dan = 0

print(f"Câu có trích dẫn (Quote = 1): {dem_cau_trich_dan} ({dem_cau_trich_dan / len(ds_cau):.1%})")

# # cau_trich_dan = []
# # for cau in ds_cau:
# #     if cau.dt6_trich_dan == 1:
# #         cau_trich_dan.append(cau)
# # in_bang(cau_trich_dan, ['cum', 'cau_sach'])

Câu có trích dẫn (Quote = 1): 4136 (23.9%)


## 4.7. Đặc trưng 7: Số lượng
sức gió 175 mph, 100,000 người, $5 million, 20%, 1st **Không tính năm** (1988, 1990s)

In [129]:
# ===== ĐẶC TRƯNG 7: SỐ LƯỢNG =====
MAU_NAM = re.compile(r'^(1[89]\d\d|20\d\d)s?$')     # năm: 1800 đến 2099, có thể có s (1990s)

def co_so_luong(cau_van, cac_tu):
    if '$' in cau_van or '%' in cau_van:          
        return 1
    for tu in cac_tu:
        co_chu_so = False
        for ky_tu in tu:
            if ky_tu.isdigit():
                co_chu_so = True
        if co_chu_so and not MAU_NAM.match(tu):
            return 1
    return 0

dem_cau_so_luong = 0
for cau in ds_cau:
    cau.dt7_so_luong = co_so_luong(cau.cau_sach, cau.cac_tu)
    dem_cau_so_luong += cau.dt7_so_luong

print(f"Câu có số lượng (dt7 = 1): {dem_cau_so_luong} ({dem_cau_so_luong / len(ds_cau):.1%})")
# # cau_so_luong = []
# # for cau in ds_cau:
# #     if cau.dt7_so_luong == 1:
# #         cau_so_luong.append(cau)
# # in_bang(cau_so_luong, ['cum', 'cau_sach'])

Câu có số lượng (dt7 = 1): 4426 (25.5%)


## 4.8. Vector đặc trưng của một câu

In [130]:
# ===== VECTOR 7 ĐẶC TRƯNG CỦA MỘT CÂU =====
# Thứ tự: 5 đặc trưng của slide 14 [Length, Fixed-Phrase, Paragraph, Thematic, Uppercase], thêm [Quote, Số lượng]
cau = ds_cau[0]                  
print("Câu:", cau.cau_sach, "\n")
print(f"(1) Length       : {cau.so_tu} từ                 -> {cau.dt1_do_dai}")
print(f"(2) Fixed-Phrase : {cau.cum_tu_tim_thay or 'không có':<50}    -> {cau.dt2_cum_tu}")
print(f"(3) Paragraph    : câu {cau.vi_tri}/{cau.so_cau_cua_bai} của bài      -> {cau.dt3_vi_tri}")
print(f"(4) Thematic     : {cau.tu_chu_de_tim_thay} -> {cau.dt4_chu_de}")
print(f"(5) Uppercase    : {cau.dt5_ten_rieng}")
print(f"(6) Quote        : {'có' if cau.dt6_trich_dan else 'không có'} ngoặc kép   -> {cau.dt6_trich_dan}")
print(f"(7) Số lượng     : {'có' if cau.dt7_so_luong else 'không có'} số lượng    -> {cau.dt7_so_luong}")
print(f"\nVector đặc trưng = [{cau.dt1_do_dai}, {cau.dt2_cum_tu}, {cau.dt3_vi_tri}, "
      f"{cau.dt4_chu_de}, {cau.dt5_ten_rieng}, {cau.dt6_trich_dan}, {cau.dt7_so_luong}]")

Câu: Hurricane Gilbert swept toward the Dominican Republic Sunday, and the Civil Defense alerted its heavily populated south coast to prepare for high winds, heavy rains and high seas. 

(1) Length       : 28 từ                 -> 1
(2) Fixed-Phrase : không có                                              -> 0
(3) Paragraph    : câu 1/16 của bài      -> INITIAL
(4) Thematic     : {'gilbert', 'coast', 'hurrican', 'wind', 'dominican'} -> 1
(5) Uppercase    : 1
(6) Quote        : không có ngoặc kép   -> 0
(7) Số lượng     : không có số lượng    -> 0

Vector đặc trưng = [1, 0, INITIAL, 1, 1, 0, 0]


## 4.9. Các đặc trưng có phân biệt được câu tóm tắt không?
So sánh tỉ lệ bật từng đặc trưng giữa câu tóm tắt (nhãn = 1) và câu thường (nhãn = 0).

In [131]:
# ===== SO SÁNH ĐẶC TRƯNG GIỮA CÂU TÓM TẮT VÀ CÂU THƯỜNG =====
ten_dac_trung = ['Length > 5 từ', 'Fixed-Phrase', 'Thematic Word', 'Uppercase Word',
                 'Paragraph = INITIAL', 'Quote', 'Số lượng']
dieu_kien = [('dt1_do_dai', 1), ('dt2_cum_tu', 1), ('dt4_chu_de', 1), ('dt5_ten_rieng', 1),
             ('dt3_vi_tri', 'INITIAL'), ('dt6_trich_dan', 1), ('dt7_so_luong', 1)]

cau_thuong, cau_tom_tat = [], []
for cau in ds_cau:
    if cau.nhan == 0:
        cau_thuong.append(cau)
    elif cau.nhan == 1:
        cau_tom_tat.append(cau)

def ti_le(danh_sach, thuc_tinh, gia_tri):
    dem = 0
    for cau in danh_sach:
        if getattr(cau, thuc_tinh) == gia_tri:
            dem += 1
    return dem / len(danh_sach) * 100

print(f"{'Đặc trưng':<22}{'Câu thường':>12}{'Câu tóm tắt':>13}")
for i in range(len(ten_dac_trung)):
    thuc_tinh, gia_tri = dieu_kien[i]
    ti_le_thuong = ti_le(cau_thuong, thuc_tinh, gia_tri)
    ti_le_tom_tat = ti_le(cau_tom_tat, thuc_tinh, gia_tri)
    print(f"{ten_dac_trung[i]:<22}{ti_le_thuong:>11.1f}%{ti_le_tom_tat:>12.1f}%")

Đặc trưng               Câu thường  Câu tóm tắt
Length > 5 từ                88.1%        99.0%
Fixed-Phrase                  3.6%        10.7%
Thematic Word                33.8%        66.8%
Uppercase Word               39.9%        63.8%
Paragraph = INITIAL           2.3%        22.8%
Quote                        24.6%         9.9%
Số lượng                     24.3%        38.9%


In [132]:
# ===== LƯU BẢNG ĐẶC TRƯNG RA CSV (mở bằng Excel được) =====
THU_MUC_DAC_TRUNG = os.path.join('.', 'DAC_TRUNG')   # nơi lưu bảng đặc trưng

os.makedirs(THU_MUC_DAC_TRUNG, exist_ok=True)
cac_cot = ['tap', 'cum', 'ma_bai', 'stt', 'cau_sach', 'so_tu', 'dt1_do_dai', 'dt2_cum_tu',
           'dt3_vi_tri', 'dt4_chu_de', 'dt5_ten_rieng', 'dt6_trich_dan', 'dt7_so_luong', 'nhan']
for tap in ['train', 'test']:
    duong_dan = os.path.join(THU_MUC_DAC_TRUNG, tap + '_features.csv')
    f = open(duong_dan, 'w', encoding='utf-8-sig', newline='')
    bo_ghi = csv.writer(f)
    bo_ghi.writerow(cac_cot)                       # dòng tiêu đề
    for cau in ds_cau:
        if cau.tap == tap:
            dong = []
            for cot in cac_cot:
                dong.append(getattr(cau, cot))
            bo_ghi.writerow(dong)
    f.close()
print("Đã lưu:", os.listdir(THU_MUC_DAC_TRUNG))

Đã lưu: ['test_features.csv', 'train_features.csv']


## 5. Phân lớp bằng Logistic Regression
**z = w0 + w1·Length + w2·FixedPhrase + w3·Initial + w4·Final + w5·Thematic + w6·Uppercase + w7·Quote + w8·SoLuong**

In [133]:
# ===== LOGISTIC REGRESSION: TẠO VECTOR, HUẤN LUYỆN, CHẤM ĐIỂM =====
CAC_DAC_TRUNG = ['dt1_do_dai', 'dt2_cum_tu', 'la_cau_dau', 'la_cau_cuoi', 'dt4_chu_de', 'dt5_ten_rieng',
                 'dt6_trich_dan', 'dt7_so_luong']

def tao_vector(cau):
    """Biến 1 câu thành 1 list số theo thứ tự CAC_DAC_TRUNG."""
    vector = []
    for dac_trung in CAC_DAC_TRUNG:
        vector.append(getattr(cau, dac_trung))
    return vector # [1, 0, 1, 0, 1, 1, 0, 0]

# Bước 1: huấn luyện
X_huan_luyen, y_huan_luyen = [], []
for cau in ds_cau:
    if cau.tap == 'train' and cau.nhan is not None:
        X_huan_luyen.append(tao_vector(cau))
        y_huan_luyen.append(cau.nhan)

mo_hinh = LogisticRegression(class_weight='balanced', max_iter=1000)
mo_hinh.fit(X_huan_luyen, y_huan_luyen)

# Bước 2: chấm điểm mọi câu = xác suất
X_tat_ca = []
for cau in ds_cau:
    X_tat_ca.append(tao_vector(cau))
xac_suat = mo_hinh.predict_proba(X_tat_ca)
# # print(xac_suat[1]) 0.07369525 0.92630475

for i in range(len(ds_cau)):
    ds_cau[i].diem_lr = xac_suat[i][1]

print(f"Huấn luyện trên {len(X_huan_luyen)} câu ({sum(y_huan_luyen)} câu tóm tắt)")
print(f"Hệ số chặn w0 = {mo_hinh.intercept_[0]:.3f}\n")
print(f"{'Đặc trưng':<18}{'Trọng số w':>12}")
trong_so = []

for i in range(len(CAC_DAC_TRUNG)):
    print(f"{CAC_DAC_TRUNG[i]:<18}{mo_hinh.coef_[0][i]:>12.3f}")

Huấn luyện trên 10497 câu (473 câu tóm tắt)
Hệ số chặn w0 = -2.500

Đặc trưng           Trọng số w
dt1_do_dai               1.871
dt2_cum_tu               0.738
la_cau_dau               2.069
la_cau_cuoi             -0.336
dt4_chu_de               0.730
dt5_ten_rieng            0.361
dt6_trich_dan           -0.984
dt7_so_luong             0.354


## 6. Lấy tóm tắt văn bản

In [134]:
# ===== CHỌN CÂU VÀ GHI FILE TÓM TẮT =====
GIOI_HAN_TU = 400 

def xep_cau_theo_uu_tien(cau):
    return (-cau.diem_lr, cau.vi_tri, -cau.so_tu) # Dấu trừ để sắp giảm dần (sorted mặc định sắp tăng dần).

def xep_lai_thu_tu_goc(cau):
    return (cau.ma_bai, cau.stt)

def tom_tat(cac_cau_cua_cum):
    xep_hang = sorted(cac_cau_cua_cum, key=xep_cau_theo_uu_tien)

    duoc_chon = []
    tong_tu = 0
    for cau in xep_hang:
        if tong_tu + cau.so_tu > GIOI_HAN_TU:
            break
        duoc_chon.append(cau)
        tong_tu += cau.so_tu

    return sorted(duoc_chon, key=xep_lai_thu_tu_goc)

THU_MUC_KET_QUA = os.path.join('.', 'KET_QUA_TOM_TAT') 
os.makedirs(THU_MUC_KET_QUA, exist_ok=True)

for cum in ds_cum:
    cum.ban_tom_tat = tom_tat(cum.ds_cau)

    f = open(os.path.join(THU_MUC_KET_QUA, cum.ma), 'w', encoding='utf-8')
    for cau in cum.ban_tom_tat:
        f.write(f'<s docid="{cau.ma_bai}" num="{cau.stt}" wdcount="{cau.so_tu}"> {cau.cau_sach}</s>\n')
    f.close()

print("Đã ghi", len(os.listdir(THU_MUC_KET_QUA)), "file tóm tắt vào", THU_MUC_KET_QUA)

Đã ghi 59 file tóm tắt vào .\KET_QUA_TOM_TAT


## 7. Đánh giá kết quả

In [135]:
# ===== ĐÁNH GIÁ: PRECISION, RECALL, F1 SO VỚI DUC_SUM =====
def dem_cau_trung(cum):
    """Số câu máy chọn của cụm có (ma_bai, stt) nằm trong DUC_SUM."""
    so_trung = 0
    for cau in cum.ban_tom_tat:
        if (cau.ma_bai, cau.stt) in cum.cau_chuyen_gia:
            so_trung += 1
    return so_trung

# Ghi kết quả vào chính object Cum; cụm có DUC_SUM rỗng thì không chấm được
cum_test, cum_train = [], []
for cum in ds_cum:
    if len(cum.dap_an) == 0:
        continue
    cum.cau_may_chon = len(cum.ban_tom_tat)          
    cum.cau_chuyen_gia_chon = len(cum.dap_an)                
    cum.trung = dem_cau_trung(cum)            
    if cum.tap == 'test':
        cum_test.append(cum)
    else:
        cum_train.append(cum)

def p_r_f1(cac_cum):
    """Cộng dồn trên nhiều cụm rồi mới chia (không lấy trung bình các %)."""
    trung, cau_may_chon, cau_chuyen_gia_chon = 0, 0, 0
    for cum in cac_cum:
        trung += cum.trung
        cau_may_chon += cum.cau_may_chon
        cau_chuyen_gia_chon += cum.cau_chuyen_gia_chon
    p = trung / cau_may_chon * 100
    r = trung / cau_chuyen_gia_chon * 100
    f1 = 2 * p * r / (p + r)
    return trung, cau_may_chon, cau_chuyen_gia_chon, p, r, f1

for ten, cac_cum in [('9 CỤM TEST (kết quả chính)', cum_test),
                     ('34 cụm train (đã dùng để huấn luyện)', cum_train)]:
    trung, cau_may_chon, cau_chuyen_gia_chon, p, r, f1 = p_r_f1(cac_cum)
    print(f"===== {ten} =====")
    print(f"Câu trùng {trung}, máy chọn {cau_may_chon} câu, chuyên gia {cau_chuyen_gia_chon} câu")
    print(f"Precision = {trung}/{cau_may_chon} = {p:.1f}%   Recall = {trung}/{cau_chuyen_gia_chon} = {r:.1f}%   F1 = {f1:.1f}%\n")

print(f"{'Cụm':<8}{'Số câu':>8}{'Máy Chọn':>10}{'CG Chọn':>10}{'Trùng':>10}{'P':>8}{'R':>8}")
for cum in cum_test:                        
    p = cum.trung / cum.cau_may_chon * 100
    r = cum.trung / cum.cau_chuyen_gia_chon * 100
    print(f"{cum.ma:<8}{len(cum.ds_cau):>8}{cum.cau_may_chon:>10}{cum.cau_chuyen_gia_chon:>10}{cum.trung:>10}{p:>8.1f}%{r:>8.1f}%")

===== 9 CỤM TEST (kết quả chính) =====
Câu trùng 29, máy chọn 119 câu, chuyên gia 123 câu
Precision = 29/119 = 24.4%   Recall = 29/123 = 23.6%   F1 = 24.0%

===== 34 cụm train (đã dùng để huấn luyện) =====
Câu trùng 141, máy chọn 466 câu, chuyên gia 515 câu
Precision = 141/466 = 30.3%   Recall = 141/515 = 27.4%   F1 = 28.7%

Cụm       Số câu  Máy Chọn   CG Chọn     Trùng       P       R
d112h        238        11        11         5    45.5%    45.5%
d113h        157        10        14         4    40.0%    28.6%
d114h        368        15        18         1     6.7%     5.6%
d115i        261        15        21         2    13.3%     9.5%
d116i        337        13        16         3    23.1%    18.8%
d117i        281        14        15         6    42.9%    40.0%
d118i        377        13        12         2    15.4%    16.7%
d119i        227        14        10         4    28.6%    40.0%
d120i        387        14         6         2    14.3%    33.3%
